# La Tour du Dernier Jour — génération gratuite des assets

Ce notebook lit `data/art/manifest.json`, `data/audio/music.json` et `tools/voice_pipeline/lines.csv`, génère sur le **GPU T4 gratuit** de Colab les **sprites des 11 personnages** (en pied, fond transparent, style de `tools/free_assets/sprite_style.json` : checkpoint + LoRA Civitai, détourage rembg), les autres images (SDXL), la musique (MusicGen) et les voix coréennes et japonaises (XTTS-v2), puis les enregistre dans `assets/` du dépôt.

**Mode d'emploi**
1. *Exécution ▸ Modifier le type d'exécution ▸* **T4 GPU** (une seule fois).
2. Facultatif : dans 🔑 *Secrets* (barre de gauche), ajouter `GITHUB_TOKEN` (jeton GitHub à grain fin, **Contents: read and write** sur le dépôt) : les fichiers sont alors poussés directement sur GitHub, et un `git pull` (ou `GENERER_ASSETS.bat`) les met en place sur ton PC. Sans jeton, un `assets_generated.zip` est téléchargé à la fin ; `GENERER_ASSETS.bat` l'installe tout seul.
   Si Civitai refuse le téléchargement des LoRA (erreur 401), ajouter aussi le secret `CIVITAI_TOKEN` (civitai.com ▸ Account Settings ▸ API Keys).
3. Cocher les réglages ci-dessous, puis **Exécution ▸ Tout exécuter**. L'**essai rapide** (cellule 1 bis) affiche d'abord Elias seul : agrandissement anime Real-ESRGAN puis seconde passe HD ×1,75 (2048 px de haut), retouche automatique du visage et des mains, image recadrée sur la silhouette. La cellule 2 bis montre ensuite la pose neutre de chaque personnage.
   Pour refaire des sprites existants (ex. après un changement de charadesign), écrire leurs identifiants dans `REGENERER` (ex. `elias`), ou `tous` après une mise à jour du style (c'est le cas de cette version : netteté HD, Aoi d'origine).

Tout est **reprenable** : si Colab se déconnecte, relancer *Tout exécuter* ; les fichiers déjà produits sont conservés (poussés par lots).

Non générées ici : les **30 CG nsfw**, réservées au backend RunPod (`python tools/generate_assets.py --paid`).
Licences des modèles : Animagine XL (usage personnel), MusicGen et XTTS-v2 (**non commercial**).

In [ ]:
#@title ⚙️ Réglages
#@markdown Essai rapide : une seule image (Elias, pose neutre, ~1 min) affichée ici, pour valider le rendu avant le lot complet.
ESSAI_ELIAS = True  #@param {type:"boolean"}
SPRITES = True  #@param {type:"boolean"}
IMAGES = True  #@param {type:"boolean"}
MUSIQUE = True  #@param {type:"boolean"}
VOIX = True  #@param {type:"boolean"}
#@markdown XTTS-v2 est sous Coqui Public Model License (usage non commercial uniquement) : cocher pour l'accepter.
ACCEPTER_LICENCE_COQUI = False  #@param {type:"boolean"}
POUSSER_SUR_GITHUB = True  #@param {type:"boolean"}
DEPOT = "iminhell/manhwa-rpg"  #@param {type:"string"}
BRANCHE = "main"  #@param {type:"string"}
#@markdown Taille des lots enregistrés (images, voix) : plus petit = moins de perte en cas de coupure.
#@markdown Sprites à refaire : identifiants séparés par des espaces (ex. « elias simone »), ou « tous » pour tout refaire
#@markdown (à faire après une mise à jour du style des sprites, si cette session Colab contient déjà des images).
REGENERER = ""  #@param {type:"string"}
LOT_SPRITES = 10  #@param {type:"integer"}
LOT_IMAGES = 12  #@param {type:"integer"}
LOT_VOIX = 150  #@param {type:"integer"}

In [ ]:
#@title 1. Préparation (GPU, dépôt, jeton)
import os, re, subprocess, sys
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
      or "⚠️ Pas de GPU : Exécution ▸ Modifier le type d'exécution ▸ T4 GPU, puis relancer.")
TOKEN = ""
try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass
CIVITAI = ""
try:
    CIVITAI = userdata.get("CIVITAI_TOKEN") or ""
except Exception:
    pass
print("Jeton GitHub :", "présent" if TOKEN else "absent (zip téléchargé à la fin)")
REPO_DIR = "/content/manhwa-rpg"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "-b", BRANCHE, f"https://github.com/{DEPOT}.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "pull", "-q", "origin", BRANCHE])
subprocess.run(["git", "config", "user.email", "colab-assets@users.noreply.github.com"])
subprocess.run(["git", "config", "user.name", "Colab (assets)"])
os.environ.update(GITHUB_TOKEN=TOKEN, CIVITAI_TOKEN=CIVITAI, DEPOT=DEPOT, BRANCHE=BRANCHE, PYTHONUNBUFFERED="1",
                  FREE_MODELS_DIR="/content/models")

def sh(*cmd):
    p = subprocess.Popen(list(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = []
    for line in p.stdout:
        print(line, end="")
        out.append(line)
    p.wait()
    if p.returncode:
        raise RuntimeError(f"échec : {' '.join(cmd)}")
    return "".join(out)

def etape(stage, lot, message):
    """Génère par lots, et enregistre (commit + push) après chaque lot, jusqu'à ce qu'il ne reste rien."""
    while True:
        out = sh(sys.executable, "tools/free_assets/free_gen.py", stage, "--limit", str(lot))
        sh(sys.executable, "tools/free_assets/publish.py", message, *(["--push"] if POUSSER_SUR_GITHUB else []))
        m = re.search(r"\] (\d+) ", out)
        if not m or int(m.group(1)) < lot:
            break

# Versions figées et testées ensemble (tools/free_assets/requirements.txt) ; torch et torchaudio restent ceux de Colab
sh("pip", "install", "-q", "-r", "tools/free_assets/requirements.txt")
sh(sys.executable, "tools/free_assets/free_gen.py", "plan")

In [ ]:
#@title 1 bis. Essai rapide : Elias (agrandissement anime ×4 → seconde passe HD ×1,75, retouche du visage et des mains, détourage)
#@markdown Génère seulement `assets/portraits/elias/neutral.png` (refait à chaque exécution) et l'affiche en taille réelle
#@markdown (zoom sur le visage pour juger la netteté). Si le rendu convient, laisser la suite s'exécuter ; sinon, interrompre
#@markdown (Exécution ▸ Interrompre) et ajuster `sprite_style.json`.
from IPython.display import Image as _Img, display
if ESSAI_ELIAS:
    sh(sys.executable, "tools/free_assets/free_gen.py", "sprites", "--force", "--only", "portrait_elias_neutral")
    if os.path.exists("assets/portraits/elias/neutral.png"):
        from PIL import Image as _PIL
        _e = _PIL.open("assets/portraits/elias/neutral.png")
        print(f"Elias : {_e.width}×{_e.height} px")
        display(_Img("assets/portraits/elias/neutral.png", height=900))
        _w = _e.width
        _e.crop((_w // 4, 0, 3 * _w // 4, _e.height // 4)).save("/content/elias_visage.png")
        display(_Img("/content/elias_visage.png"))  # visage à 100 % : le trait doit être net
    else:
        print("⚠️ Elias n'a pas été enregistré (voir les messages « ! » ci-dessus) : ne pas lancer le lot complet.")

In [ ]:
#@title 2. Sprites des personnages (en pied, fond transparent · environ 40 s par sprite)
#@markdown Pose neutre générée une fois par personnage, puis chaque expression par inpainting du visage
#@markdown (corps identique d'une expression à l'autre), détourage rembg isnet-anime → `assets/portraits/` et `assets/sprites/`.
if SPRITES:
    if REGENERER.strip() == "tous":
        sh(sys.executable, "tools/free_assets/free_gen.py", "sprites", "--force")
        sh(sys.executable, "tools/free_assets/publish.py", "Assets : tous les sprites refaits",
           *(["--push"] if POUSSER_SUR_GITHUB else []))
    elif REGENERER.split():
        sh(sys.executable, "tools/free_assets/free_gen.py", "sprites", "--force", "--only", *REGENERER.split())
        sh(sys.executable, "tools/free_assets/publish.py", f"Assets : sprites refaits ({REGENERER})",
           *(["--push"] if POUSSER_SUR_GITHUB else []))
    etape("sprites", LOT_SPRITES, "Assets : sprites de personnages (Colab, SDXL + LoRA, fond transparent)")

In [ ]:
#@title 2 bis. Aperçu des sprites (pose neutre de chaque personnage, et personnages manquants)
from IPython.display import HTML, display
import base64, glob, json
_tuiles = []
for _p in sorted(glob.glob("assets/portraits/*/neutral.png")):
    _b64 = base64.b64encode(open(_p, "rb").read()).decode()
    _tuiles.append(f'<figure style="display:inline-block;margin:6px;text-align:center"><img src="data:image/png;base64,{_b64}" '
                   f'style="height:420px;background:#2b2b2b"><figcaption>{_p.split("/")[2]}</figcaption></figure>')
display(HTML("".join(_tuiles) or "Aucun sprite pour l'instant."))
_persos = sorted({a["char"] for a in json.load(open("data/art/manifest.json"))["assets"] if a["kind"] == "portrait"})
_manque = [c for c in _persos if not os.path.exists(f"assets/portraits/{c}/neutral.png")]
print("Personnages sans pose neutre :", ", ".join(_manque) if _manque else "aucun")

In [ ]:
#@title 3. Autres images : décors, ennemis, CG (SDXL · environ 30 s par image)
if IMAGES:
    etape("images", LOT_IMAGES, "Assets : images générées (Colab, SDXL)")

In [ ]:
#@title 4. Musique (MusicGen · environ 1 min par piste)
if MUSIQUE:
    etape("music", 6, "Assets : musique générée (Colab, MusicGen)")

In [ ]:
#@title 5. Voix coréennes et japonaises (XTTS-v2 · environ 2 s par réplique)
if VOIX and not ACCEPTER_LICENCE_COQUI:
    print("Voix ignorées : cocher ACCEPTER_LICENCE_COQUI dans les réglages, puis relancer.")
elif VOIX:
    os.environ["COQUI_TOS_AGREED"] = "1"
    etape("voices", LOT_VOIX, "Assets : voix générées (Colab, XTTS-v2)")

In [ ]:
#@title 6. Livraison
sh(sys.executable, "tools/free_assets/free_gen.py", "plan")
if POUSSER_SUR_GITHUB and TOKEN:
    print(f"✅ Tout est sur GitHub ({DEPOT}@{BRANCHE}). Sur le PC : git pull, ou double-clic sur GENERER_ASSETS.bat.")
else:
    sh(sys.executable, "tools/free_assets/publish.py", "Assets générés (Colab)", "--zip")
    from google.colab import files
    files.download("assets_generated.zip")
    print("✅ assets_generated.zip téléchargé : double-clic sur GENERER_ASSETS.bat pour l'installer dans assets/.")